# Intermediate Data Science

## Important Information

- Email: [joanna_bieri@redlands.edu](mailto:joanna_bieri@redlands.edu)
- Office Hours take place in Duke 209 -- [Office Hours Schedule](https://joannabieri.com/schedule.html)
- [Class Website](https://joannabieri.com/data201_intermediate.html)
- [Syllabus](https://joannabieri.com/data201/IntermediateDataScience.pdf)

## Today's Reading

*Python for Data Analysis*, Chapter 8 - Data Wrangling: Join, Combine, and Reshape. The notes below follow the book fairly closely, so keep a notebook open and try the commands as you go.

## Career Discussion - today

We are talking about *Build a Career in Data Science*, 1.2 Different Types of Data Science Jobs at the start of class. Come with your notes on the three kinds of data science jobs the book describes.

## Data Wrangling

Data Wrangling is the art of managing data that might be spread across many files or databases. It also involves organizing data that comes to you in an inconvenient format. We are going to explore some ways that Pandas can help us in organizing our data!


In [ ]:
# Some basic package imports
import os
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = 'notebook_connected'

## Hierarchical Indexing

Hierarchical indexing is a feature of pandas that allows you to have more than one index on a single axis in a DataFrame. This is like working with data in a table but allowing it to be higher dimensional. Here is the example from our book:

In [ ]:
# This series is a list of lists
data = pd.Series(np.random.uniform(size=9),
                 index=[["a", "a", "a", "b", "b", "c", "c", "d", "d"],
                        [1, 2, 3, 1, 3, 1, 2, 2, 3]])
data

In [ ]:
data.index

Notice here that the index now has two dimensions. You can reach in and grab the stuff inside the 'a' grouping and then within that choose 1,2,3. We can use partial indexing to grab different parts of the data.

In [ ]:
# To get a specific entry you select both index values
data['a',1]

In [ ]:
# You can grab all of an outer index
data['a']

In [ ]:
# You can grab multiple outer index values
data[['a','c']]

In [ ]:
# You can select from an inner level index with a : meaning all of the outer index values
data.loc[:,1]

Let's see what happens when we put this data into a DataFrame. Notice how we have only one column and hierarchical indexing on the left side.

In [ ]:
pd.DataFrame(data)

We can use the `.unstack()` function to send this data into a regular DataFrame. Unstack takes the inner index and sends it into separate columns, while keeping the outer index as the DataFrame index.


In [ ]:
data.unstack()

You can also `.stack()` a DataFrame to send the columns back into a hierarchical index.


In [ ]:
data.unstack().stack()

In a DataFrame either the columns or the rows can have hierarchical levels.

In [ ]:
frame = pd.DataFrame(np.arange(12).reshape((4, 3)),
                     index=[["a", "a", "b", "b"], [1, 2, 1, 2]],
                     columns=[["Ohio", "Ohio", "Colorado"],
                              ["Green", "Red", "Green"]])
frame

In [ ]:
# The keys are two dimensional
frame.keys()

In [ ]:
# The indices are two dimensional
frame.index

In [ ]:
# How do I get data out of this thing?
frame['Ohio','Green']

In [ ]:
frame['Ohio','Green'].loc['a',2]

In [ ]:
# Right now the columns don't have names
frame.columns.names

In [ ]:
# And the index labels don't have names
frame.index.names

In [ ]:
# Lets add some names
frame.index.names = ["key1", "key2"]
frame.columns.names = ["state", "color"]
frame

## You try

What does frame.unstack() do in this case. Go ahead and run the command and see if you can understand the results.

In [ ]:
# Code and/or words here

## Reordering and Sorting Levels

In the example above the levels were state then color and a/b then number. So it is easy to select by the outer level, and a bit harder to get the inside levels.

In [ ]:
frame['Ohio']

In [ ]:
# This would give a key error because it expects outer level first
# frame['Green']

So if we wanted to look at just the green data, we would need to reorder the levels, making color outer and state inner.

In [ ]:
new_frame = frame.swaplevel('state','color', axis=1)
new_frame

In [ ]:
new_frame['Green']

## You Try

How would you swap the index keys? See if you can swap key1 and key2 in the new_frame.

In [ ]:
# Your code here

## Summary Statistics by Level

If we run statistics on the DataFrame as a whole, it ignores the levels and does the operation to the whole thing. This might be okay in some instances, but in the data frame above maybe we want to sum the "a" and "b" groupings separately.

In [ ]:
# Good old fashioned sum
frame.sum()

In [ ]:
# Grouped sum
frame.groupby(level='key1').sum()

The grouped sum lets us look at the sums of the specific index levels. We could also group by the columns! If we just transpose the data frame then the column names become the index names!

In [ ]:
frame.T

In [ ]:
frame.T.groupby(level="color").sum()

## Combining and Merging Datasets

Sometimes in your analysis you will want to grab data from more than one file, or maybe you scrape data from more than one website. In these cases you need to be able to merge the data into a single dataset for analysis. There are a few great Pandas commands for this:

- pd.merge() - connects the rows in separate DataFrames based on one or more keys. It implements the database join operations.
- pd.concat() - this is short for concatenate. Concatenate stacks objects along an axis. For example stacking rows to add more observations or stacking columns to add more variables to the existing observations.
- combine_first() - splices together overlapping data to fill in missing values in one object with values from another.


## Pandas Merge

Merge connects separate DataFrames based on comparing keys (or column labels). There are different merge types available:

- `inner` is the most restrictive and only includes cases where the keys match across both datasets.
- `left` includes all entries in the left dataset and only those that match from the right dataset.
- `right` includes all entries in the right dataset and only those that match from the left dataset.
- `outer` includes all entries in both datasets.

We will start with two example DataFrames and explore the results for the different merge types.

In [ ]:
employees = pd.DataFrame({
    'employee_id': [1, 2, 3, 4, 5],
    'name': ['Alice', 'Bob', 'Charlie', 'Diana', 'Eve'],
    'department_id': [10, 20, 10, 30, 99]  # Eve is in unknown dept 99
})

# Departments DataFrame (has an extra dept with no employees)
departments = pd.DataFrame({
    'department_id': [10, 20, 30, 40],
    'department_name': ['Engineering', 'HR', 'Marketing', 'Sales']  # Dept 40 has no employees
})

display(employees)
display(departments)

Looking at these two data sets we see that there are 5 employees and 4 departments. The two DataFrames share the key department_id. **You need a shared key or shared data to merge!** You will also notice some mismatch between the datasets. For example none of our employees have the department_id 40=Sales, and one of our employees has a department_id 99, which does not appear in our departments data frame. 

Lets look at the merges below. Note: we are using employees and the left and departments as the right dataset. This could be switched. Both datasets have the department_id key!

In [ ]:
# Inner - THIS IS DEFAULT if you don't choose how
# What data is missing? Do you see any Nan?
pd.merge(employees, departments, on='department_id', how='inner')

In [ ]:
# Left
# What data is missing? Do you see any Nan?
pd.merge(employees, departments, on='department_id', how='left')

In [ ]:
# Right
# What data is missing? Do you see any Nan?
pd.merge(employees, departments, on='department_id', how='right')

In [ ]:
# Outer
# What data is missing? Do you see any Nan?
pd.merge(employees, departments, on='department_id', how='outer')

| Merge Type | Includes All Employees | Includes All Departments | Notes               |
|------------|------------------------|---------------------------|---------------------|
| Inner      | ❌ Only matched         | ❌ Only matched           | Most restrictive    |
| Left       | ✅ Yes                 | ❌ Only matched           | Focus on employees  |
| Right      | ❌ Only matched         | ✅ Yes                   | Focus on departments|
| Outer      | ✅ Yes                 | ✅ Yes                   | Full outer view     |

How would you merge data sets if one had the correct data, but did not have the correct key? Well, one option would be to change the column labels to match, but you could also tell pandas.merge() two different keys.

In [ ]:
departments.rename(columns={'department_id':'dept_code'}, inplace=True)
display(departments)

In [ ]:
pd.merge(employees, departments, left_on='department_id', right_on='dept_code')

You will notice that when the merge happens it resets the index. If you want to preserve the index values for each you may want to store the index values as a column in the data set. This will allow you to look up the original values in the separate data sets if needed.

Sometimes you need to use index values to do your merge. Maybe the matching data is in the index instead of a separate column.

In [ ]:
departments = pd.DataFrame({
    'department_name': ['Engineering', 'HR', 'Marketing']
}, index=[10, 20, 30])

display(departments)

In [ ]:
pd.merge(employees,departments, left_on='department_id', right_index=True)

## You Try

Merge the following data sets using all four ways: inner, left, right, and outer. See if you can predict before running the code what the output will be!

In [ ]:
df_animals = pd.DataFrame({
    'animal_id': [1, 2, 3, 4],
    'name': ['Leo', 'Stripes', 'Spot', 'Fluffy'],
    'type': ['Lion', 'Tiger', 'Cheetah', 'Cat']
})

df_habitats = pd.DataFrame({
    'animal_id': [1, 2, 5, 4],
    'habitat': ['Savannah', 'Jungle', 'Mountains', 'Domestic'],
    'population_estimate': [25000, 3200, 120, 50000000]
})

display(df_animals)
display(df_habitats)

In [ ]:
# Your prediction here

In [ ]:
# Your code here

In [ ]:
# Repeat

### Hierarchical index values:

When you have hierarchical index values or columns, things get more confusing but merges are still possible!

In [ ]:
left_data = pd.DataFrame({"key1": ["Ohio", "Ohio", "Ohio",
                               "Nevada", "Nevada"],
                      "key2": [2000, 2001, 2002, 2001, 2002],
                      "data": pd.Series(range(5), dtype="Int64")})
right_data_index = pd.MultiIndex.from_arrays(
    [
        ["Nevada", "Nevada", "Ohio", "Ohio", "Ohio", "Ohio"],
        [2001, 2000, 2000, 2000, 2001, 2002]
    ]
)

right_data = pd.DataFrame({"event1": pd.Series([0, 2, 4, 6, 8, 10], dtype="Int64",
                                           index=right_data_index),
                       "event2": pd.Series([1, 3, 5, 7, 9, 11], dtype="Int64",
                                           index=right_data_index)})

display(right_data)
display(left_data)

Now we need to indicate multiple columns to merge on as a list.

In [ ]:
pd.merge(left_data, right_data, left_on=["key1", "key2"], right_index=True)

## Concatenate

When you need to stack new rows or columns onto an existing data set pd.concat() is a great way to do that. 

Let's imagine that we are working with the employee and department information above. Now suddenly HR sends us information about two new employees and data that contains all the salaries. They have confirmed that the salaries are in increasing order of the employee id. How do we get all this data into a single dataframe?

1. Concat the new_hires onto the employees data
2. Merge the employees and department data - keeping all information
3. Concat the new columns onto the full data set.

When using concat the dimensions must match!

- `axis=0` must have the same number of columns - you are adding rows
- `axis=1` must have the same number of rows - you are adding columns

In [ ]:
new_hires = pd.DataFrame({
    'employee_id': [6, 7],
    'name': ['Joanna', 'Bella'],
    'department_id': [30, 20]
})

salaries = pd.DataFrame({
    'emp_num': ['emp_'+str(i+1) for i in range(7)],
    'salary': [60_000, 55_000, 62_000, 58_000, 500_000, 40_000, 40_000]
})

display(new_hires)
display(salaries)

In [ ]:
# Concat the new hires
# Here we ignore the old index values and reindex so the rows are 0-6
all_employees = pd.concat([employees, new_hires],ignore_index=True)
display(all_employees)

In [ ]:
# Merge the department data
all_employees = pd.merge(all_employees,departments,left_on='department_id',right_index=True,how='outer')
display(all_employees)

In [ ]:
# Concat the salaries
full_data = pd.concat([all_employees, salaries], axis=1)
display(full_data)

## Combining Data with Overlap

Sometimes you have two datasets that have an overlap, but one or both of them are incomplete and you want to use one to fill in NaNs in the other. You can think of the `combine_first()` operation as patching up the data. It basically does an if-else statement that inserts values if there are null values in the original dataset.

Here is a scenario where maybe you have incomplete employee data. However each of the datasets has missing data and you want one complete dataset.

In [ ]:
employee_profiles = pd.DataFrame({
    'name': ['Alice', None, 'Charlie'],
    'dept_code': [10, 20, None],
    'email': [None, 'bob@example.com', None]
}, index=[1, 2, 3])  

backup_profiles = pd.DataFrame({
    'name': ['Alice A.', 'Bob B.', 'Charlie C.', 'Diana D.'],
    'dept_code': [10, 20, 10, 30],
    'email': ['alice@example.com', None, 'charlie@example.com', 'diana@example.com'],
    'phone': ['111-1111', '222-2222', '333-3333', '444-4444']  
}, index=[1, 2, 3, 4]) 

display(employee_profiles)
display(backup_profiles)

In [ ]:
combined_profiles = employee_profiles.combine_first(backup_profiles)
display(combined_profiles)

combine_first() does not overwrite existing data in the first DataFrame. It aligns on both index and column names, and mismatches are handled gracefully. You can think of it as a data patching tool.

**combine_first() is perfect when:**

- You have a primary source of data that may be incomplete.
- You have a secondary or backup source you want to use to fill in the blanks.
- You want row-wise alignment based on index.

But what happens if the indexes don't align?

## Reshaping and Pivoting

The goal with reshaping and pivoting is to rearrange tabular data in a way that is better for your specific analysis. We have already seen some of these commands.

- `stack()` rotates or pivots from the columns in the data to the rows when provided hierarchical indexes.
- `unstack()` rotates or pivots from the rows into the columns creating hierarchical indexes.
- `.pivot()` reshapes the data from long(tall) format to a wide format.
- `melt()` reshapes the data from wide to long(tall) melting the column names into the data

**Long (Tall) Format** Also called "tidy" data in some contexts:

- One row per observation.
- Repeated categories or measurements in a single column.
- More rows, fewer columns.
- One row per person per test

**Wide Format**

- Unique values in a categorical column become column headers.
- More columns, fewer rows.
- Easier for humans to read, but not always ideal for statistical analysis.
- One row per person, one column per test

Below we will look at the ways we might rearrange or reshape our data!

In [ ]:
df = pd.DataFrame({
    'person': ['Alice', 'Alice', 'Bob', 'Bob'],
    'month': ['Jan', 'Feb', 'Jan', 'Feb'],
    'sales': [200, 180, 210, 190],
    'expenses': [150, 120, 160, 140]
})

display(df)

### Pivot

When you pivot a DataFrame you tell it which columns to use for the new data set

- `index` which column should be used as the new row (index) labels.
- `column` which column should be used as the new column labels.
- `values` which column should be used to fill in values in the new data set.

In [ ]:
pivoted = df.pivot(index='person',columns='month', values='sales')
display(pivoted)

In [ ]:
pivoted = df.pivot(index='month',columns='person', values='expenses')
display(pivoted)

In [ ]:
# You can specify more than one value to be added as hierarchical columns
pivoted = df.pivot(index='person',columns='month', values=['sales','expenses'])
display(pivoted)

## You Try

Do a pivot on your merged animal data. You can decide how to pivot, but try to say before running the code what you expect to happen.

In [ ]:
# Your prediction here

In [ ]:
# Your code here

### Stack

Stack takes the column labels and moves them into the index, shifting the data from wide to tall format. You will see one grouping of data for each index. In many cases you might want to set the index values to be more interesting to get a better breakdown of the data.

We go from wide data to tall data, meaning one observation per index per column.


In [ ]:
# Stack without setting the index
stacked = df.stack()
display(stacked)

In [ ]:
# Stack with index set
stacked = df.set_index(['person', 'month']).stack()
display(stacked)

In [ ]:
# Adding the .reset_index() command to a series will send it to a dataframe again
# notice how the stacking is preserved in the row order.
stacked = df.set_index(['person', 'month']).stack().reset_index()
display(stacked)

### Unstack

Lets say you are given data with observations for each person. But what you want is a wide data frame, with fewer rows and more categorical columns. This is what unstack can do!

Let's start with some stacked data - hierarchical indexes

In [ ]:
stacked = df.set_index(['person','month']).stack()
display(stacked)

In [ ]:
# By default unstack moves the innermost index level into the columns.
# Here that is the sales/expenses level, so those become the columns again.
stacked.unstack()

In [ ]:
# We can specify which level to use
# Here use the Person level index as the columns
stacked.unstack(level=0)

In [ ]:
# Here use the month level index as the columns
stacked.unstack(level=1)

In [ ]:
# Here use the innermost values as the columns
stacked.unstack(level=2)

### Melt

The melt command lets you choose a column (or use all columns) to be used as an additional row in the data.

In [ ]:
df

In [ ]:
# Use all the columns - not so useful in this case!
pd.melt(df)

In [ ]:
# Melt to create new rows for each observation for each person each month
# You keep the columns 'person' and 'month' the rest are melted
pd.melt(df,id_vars=['person','month'])

## Summary

Merging, concatenating, and reshaping are the tools you reach for when the data you need is not all in one place, or is in the wrong shape for the question you want to ask. A huge percentage of your time as a data scientist is spent right here.

## Homework 5

The full assignment is in `HW_day5.ipynb` in your sandbox. The short version: three phone usage datasets that have to be merged before you can ask whether usage differs by device. Decide which columns are the keys, decide which merge type you need and say why, then ask at least three questions of the merged data.

Work the problems in your sandbox. Your team's write-up notebook goes in `Week03` of your team repo. Homework 5 and Homework 6 are due together, Sunday 9/20 at 11:59pm.
